# LLM Agent Orchestration with LangChain and Granite

This notebook demonstrates how to build an LLM agent orchestration pipeline using IBM® watsonx.ai Granite model and LangChain. The pipeline processes a text document (The Adventures of Sherlock Holmes), creates a vector index for semantic search, and answers user queries using Retrieval-Augmented Generation (RAG).

**Reference:** https://www.ibm.com/es-es/think/tutorials/llm-agent-orchestration-with-langchain-and-granite

## Step 3. Install Required Packages

Install the essential libraries needed to work with LangChain and IBM® watsonx.ai.

In [ ]:
!pip install --upgrade pip

!pip install langchain faiss-cpu pandas sentence-transformers

%pip install langchain

!pip install langchain-ibm

### Package descriptions

- **`langchain`**: The core framework for building applications with language models.
- **`faiss-cpu`**: For efficient similarity search, used to create and query vector indexes.
- **`pandas`**: For data manipulation and analysis.
- **`sentence-transformers`**: To generate embeddings for semantic search.
- **`langchain-ibm`**: To integrate IBM watsonxLLM (`ibm/granite-3-8b-instruct`) with LangChain.

## Step 4. Import Necessary Libraries

In [ ]:
import os

from langchain_ibm import WatsonxLLM

from langchain.embeddings import HuggingFaceEmbeddings

from langchain.vectorstores import FAISS

from langchain.text_splitter import RecursiveCharacterTextSplitter

import pandas as pd

import getpass

### Module descriptions

- **`os`**: Provides a way to interact with the operating system (e.g., accessing environment variables).
- **`langchain_ibm.WatsonxLLM`**: Allows using IBM® watsonx Granite LLM seamlessly within the LangChain framework.
- **`langchain.embeddings.HuggingFaceEmbeddings`**: Generates embeddings for text using HuggingFace models, essential for semantic search.
- **`langchain.vectorstores.FAISS`**: A library for efficient vector storage and similarity search.
- **`RecursiveCharacterTextSplitter`**: Helps split large blocks of text into smaller chunks.
- **`pandas`**: A powerful library for data analysis and manipulation.
- **`getpass`**: A secure way to capture sensitive input (e.g., API keys) without displaying them on screen.

## Step 5. Configure Credentials

Set up credentials to access the IBM® watsonx Machine Learning (WML) API.

In [ ]:
# Set up credentials
credentials = {
    "url": "https://us-south.ml.cloud.ibm.com",  # Replace with the correct region if needed
    "apikey": getpass.getpass("Please enter your WML API key (hit enter): ")
}

# Set up project_id
try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = input("Please enter your project_id (hit enter): ")

## Step 6. Initialize the Large Language Model

Initialize IBM watsonxLLM using the `ibm/granite-3-8b-instruct` model.

In [ ]:
# Initialize the IBM Granite LLM
llm = WatsonxLLM(
    model_id="ibm/granite-3-8b-instruct",
    url=credentials["url"],
    apikey=credentials["apikey"],
    project_id=project_id,
    params={
        "max_new_tokens": 150
    }
)

## Step 7. Define a Function to Extract Text from a File

This function reads and extracts the content of a plain text file.

In [ ]:
def extract_text_from_txt(file_path):
    """Extracts text from a plain text file."""
    with open(file_path, "r", encoding="utf-8") as file:
        text = file.read()
    return text

## Step 8. Split the Text into Chunks

Split large blocks of text into smaller, manageable chunks for efficient processing and indexing.

In [ ]:
def split_text_into_chunks(text, chunk_size=500, chunk_overlap=50):
    """Splits text into smaller chunks for indexing."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_text(text)

## Step 9. Create a Vector Index

Convert text chunks into embeddings and store them in a FAISS vector index for semantic search.

In [ ]:
def create_vector_index(chunks):
    """Creates a FAISS vector index from text chunks."""
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = FAISS.from_texts(chunks, embeddings)
    return vector_store

## Step 10. Query the Vector Index with Granite

Query the vector index to retrieve relevant information and use IBM Granite LLM to generate a refined response.

In [ ]:
def query_index_with_granite_dynamic(vector_store, query, llm):
    """Searches the vector index, uses Granite to refine the response, and returns all components."""
    # Perform similarity search
    print("\n> Entering new AgentExecutor chain...")
    thought = f"The query '{query}' requires context from the book to provide an accurate response."
    print(f" Thought: {thought}")
    action = "Search FAISS Vector Store"
    print(f" Action: {action}")
    action_input = query
    print(f" Action Input: \"{action_input}\"")

    # Retrieve context
    results = vector_store.similarity_search(query, k=3)
    observation = "\n".join([result.page_content for result in results])
    print(f" Observation:\n{observation}\n")

    # Generate response with Granite
    prompt = f"Context:\n{observation}\n\nQuestion: {query}\nAnswer:"
    print(f" Thought: Combining retrieved context with the query to generate a detailed answer.")
    final_answer = llm.invoke(prompt)
    print(f" Final Answer: {final_answer.strip()}")
    print("\n> Finished chain.")

    # Return all components as a dictionary
    return {
        "Thought": thought,
        "Action": action,
        "Action Input": action_input,
        "Observation": observation,
        "Final Answer": final_answer.strip()
    }

## Step 11. Generate a DataFrame for Query Results

Process multiple queries dynamically, retrieve relevant information, and save the results in a structured format.

In [ ]:
def dynamic_output_to_dataframe(vector_store, queries, llm, csv_filename="output.csv"):
    """Generates a DataFrame dynamically for multiple queries and saves it as a CSV file."""
    # List to store all query outputs
    output_data = []

    # Process each query
    for query in queries:
        # Capture the output dynamically
        output = query_index_with_granite_dynamic(vector_store, query, llm)
        output_data.append(output)

    # Convert the list of dictionaries into a DataFrame
    df = pd.DataFrame(output_data)

    # Display the DataFrame
    print("\nFinal DataFrame:")
    print(df)

    # Save the DataFrame as a CSV file
    df.to_csv(csv_filename, index=False)
    print(f"\nOutput saved to {csv_filename}")

## Step 12. Run the Main Workflow

Combine all previous steps into a single workflow to process the text file, answer user queries, and save results in a structured format.

> **Note:** Make sure the file `aosh.txt` (The Adventures of Sherlock Holmes) is available in the same directory as this notebook. You can download the text from [Project Gutenberg](https://www.gutenberg.org/ebooks/1661).

In [ ]:
def main_workflow():
    # Replace with your text file
    file_path = "aosh.txt"

    # Extract text from the text file
    text = extract_text_from_txt(file_path)

    # Split the text into chunks
    chunks = split_text_into_chunks(text)

    # Create a vector index
    vector_store = create_vector_index(chunks)

    # Define queries
    queries = [
        "What is the plot of 'A Scandal in Bohemia'?",
        "Who is Dr. Watson, and what role does he play in the stories?",
        "Describe the relationship between Sherlock Holmes and Irene Adler.",
        "What methods does Sherlock Holmes use to solve cases?"
    ]

    # Generate and save output dynamically
    dynamic_output_to_dataframe(vector_store, queries, llm)


# Run the workflow
main_workflow()